In [68]:
import numpy as np
import matplotlib.pyplot as plt
import os
import urllib.request

np.random.seed(42)
print("NumPy version:", np.__version__)

NumPy version: 2.0.2


In [69]:
url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
fname = "mnist.npz"

if not os.path.exists(fname):
    print("Downloading MNIST...")
    urllib.request.urlretrieve(url, fname)
    print("Done!")

data    = np.load(fname)
X_train = data["x_train"].reshape(-1, 784) / 255.0
Y_train = data["y_train"]
X_test  = data["x_test"].reshape(-1, 784)  / 255.0
Y_test  = data["y_test"]

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_test: ", X_test.shape)
print("Y_test: ", Y_test.shape)

# Convert labels to one-hot vectors
def one_hot(y, num_classes=10):
    oh = np.zeros((len(y), num_classes))
    oh[np.arange(len(y)), y] = 1
    return oh

Y_train_oh = one_hot(Y_train)
Y_test_oh  = one_hot(Y_test)
print("Y_train one-hot:", Y_train_oh.shape)

X_train: (60000, 784)
Y_train: (60000,)
X_test:  (10000, 784)
Y_test:  (10000,)
Y_train one-hot: (60000, 10)


In [70]:
def init_weights():
    np.random.seed(42)
    W1 = np.random.uniform(-0.5, 0.5, (784, 128))
    b1 = np.zeros((1, 128))
    W2 = np.random.uniform(-0.5, 0.5, (128, 64))
    b2 = np.zeros((1, 64))
    W3 = np.random.uniform(-0.5, 0.5, (64, 10))
    b3 = np.zeros((1, 10))
    return W1, b1, W2, b2, W3, b3

W1, b1, W2, b2, W3, b3 = init_weights()

total = W1.size + b1.size + W2.size + b2.size + W3.size + b3.size
print("W1:", W1.shape, "  b1:", b1.shape)
print("W2:", W2.shape, "  b2:", b2.shape)
print("W3:", W3.shape, "  b3:", b3.shape)
print("Total parameters", total)

W1: (784, 128)   b1: (1, 128)
W2: (128, 64)   b2: (1, 64)
W3: (64, 10)   b3: (1, 10)
Total parameters 109386


In [71]:
def sigmoid(z):
    # clip prevents overflow for very large/small z values
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    # derived in Task 3: sigma'(z) = a(1 - a)
    return a * (1 - a)

# Sanity checks
print("sigmoid(0)  =", sigmoid(0))          # should be 0.5
print("sigmoid(1)  =", round(sigmoid(1), 4))  # should be 0.7311
print("sigmoid(-1) =", round(sigmoid(-1), 4)) # should be 0.2689
print("max of sigmoid_derivative ", sigmoid_derivative(0.5))  # should be 0.25

sigmoid(0)  = 0.5
sigmoid(1)  = 0.7311
sigmoid(-1) = 0.2689
max of sigmoid_derivative  0.25


In [72]:
def forward_pass(X, W1, b1, W2, b2, W3, b3):
    # Hidden Layer 1
    Z1 = X @ W1 + b1        # shape: (m, 128)
    A1 = sigmoid(Z1)         # shape: (m, 128)

    # Hidden Layer 2
    Z2 = A1 @ W2 + b2        # shape: (m, 64)
    A2 = sigmoid(Z2)         # shape: (m, 64)

    # Output Layer
    Z3 = A2 @ W3 + b3        # shape: (m, 10)
    A3 = sigmoid(Z3)         # shape: (m, 10)

    return Z1, A1, Z2, A2, Z3, A3

# Test on 5 samples
Z1t, A1t, Z2t, A2t, Z3t, A3t = forward_pass(
    X_train[:5], W1, b1, W2, b2, W3, b3)

print("Forward pass output shapes:")
print("  Z1:", Z1t.shape, "  A1:", A1t.shape)
print("  Z2:", Z2t.shape, "  A2:", A2t.shape)
print("  Z3:", Z3t.shape, "  A3:", A3t.shape)

Forward pass output shapes:
  Z1: (5, 128)   A1: (5, 128)
  Z2: (5, 64)   A2: (5, 64)
  Z3: (5, 10)   A3: (5, 10)


In [73]:
def mse_loss(Y_true, Y_pred):
    m = Y_true.shape[0]
    return np.sum((Y_true - Y_pred) ** 2) / m

# Check initial loss before any training
_, _, _, _, _, A3_init = forward_pass(
    X_train, W1, b1, W2, b2, W3, b3)

initial_loss = mse_loss(Y_train_oh, A3_init)
print("Initial MSE loss", round(initial_loss, 6))

Initial MSE loss 3.041416


In [74]:
def backpropagation(X, Y_true, Z1, A1, Z2, A2, Z3, A3,
                    W1, W2, W3):
    m = X.shape[0]

    # --- Output layer delta ---
    # delta3 = dL/dŷ * sigmoid'(A3) = -2(y - ŷ) * A3(1 - A3)
    delta3 = -2 * (Y_true - A3) * sigmoid_derivative(A3)  # (m, 10)

    # Gradients for W3 and b3
    dW3 = (A2.T @ delta3) / m                              # (64, 10)
    db3 = np.sum(delta3, axis=0, keepdims=True) / m        # (1, 10)

    # --- Hidden layer 2 delta ---
    delta2 = (delta3 @ W3.T) * sigmoid_derivative(A2)     # (m, 64)

    # Gradients for W2 and b2
    dW2 = (A1.T @ delta2) / m                              # (128, 64)
    db2 = np.sum(delta2, axis=0, keepdims=True) / m        # (1, 64)

    # --- Hidden layer 1 delta ---
    delta1 = (delta2 @ W2.T) * sigmoid_derivative(A1)     # (m, 128)

    # Gradients for W1 and b1
    dW1 = (X.T @ delta1) / m                               # (784, 128)
    db1 = np.sum(delta1, axis=0, keepdims=True) / m        # (1, 128)

    return dW1, db1, dW2, db2, dW3, db3

# Quick shape test
Z1t,A1t,Z2t,A2t,Z3t,A3t = forward_pass(
    X_train[:32], W1, b1, W2, b2, W3, b3)
grads = backpropagation(
    X_train[:32], Y_train_oh[:32],
    Z1t, A1t, Z2t, A2t, Z3t, A3t, W1, W2, W3)
names = ["dW1","db1","dW2","db2","dW3","db3"]
for n, g in zip(names, grads):
    print(f"  {n}: {g.shape}")

  dW1: (784, 128)
  db1: (1, 128)
  dW2: (128, 64)
  db2: (1, 64)
  dW3: (64, 10)
  db3: (1, 10)


In [75]:
def update_weights(W1, b1, W2, b2, W3, b3,
                   dW1, db1, dW2, db2, dW3, db3,
                   learning_rate):
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    W3 = W3 - learning_rate * dW3
    b3 = b3 - learning_rate * db3
    return W1, b1, W2, b2, W3, b3

print("update_weights function defined.")

update_weights function defined.


In [ ]:
learning_rate = 0.1
epochs        = 20
batch_size    = 32
loss_history  = []

# Reset weights to fresh start
W1, b1, W2, b2, W3, b3 = init_weights()

print("Training...\n")
print(f"{'Epoch':>6} | {'MSE Loss':>12}")
print("-" * 22)

for epoch in range(epochs):

    # Shuffle training data at start of each epoch
    idx    = np.random.permutation(X_train.shape[0])
    X_shuf = X_train[idx]
    Y_shuf = Y_train_oh[idx]

    # Mini-batch loop
    for start in range(0, X_train.shape[0], batch_size):
        X_batch = X_shuf[start : start + batch_size]
        Y_batch = Y_shuf[start : start + batch_size]

        # Forward pass
        Z1, A1, Z2, A2, Z3, A3 = forward_pass(
            X_batch, W1, b1, W2, b2, W3, b3)

        # Backpropagation
        dW1, db1_g, dW2, db2_g, dW3, db3_g = backpropagation(
            X_batch, Y_batch,
            Z1, A1, Z2, A2, Z3, A3,
            W1, W2, W3)

        # Weight update
        W1, b1, W2, b2, W3, b3 = update_weights(
            W1, b1, W2, b2, W3, b3,
            dW1, db1_g, dW2, db2_g, dW3, db3_g,
            learning_rate)

    # Compute and record full training loss after each epoch
    _, _, _, _, _, A3_full = forward_pass(
        X_train, W1, b1, W2, b2, W3, b3)
    epoch_loss = mse_loss(Y_train_oh, A3_full)
    loss_history.append(epoch_loss)
    print(f"{epoch+1:>6} | {epoch_loss:>12.6f}")

print("\nTraining complete!")

Training...

 Epoch |     MSE Loss
----------------------
     1 |     0.295728


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs + 1), loss_history,
         "o-", color="#534AB7", linewidth=2, markersize=6)
plt.title("Training Loss Curve — MLP on MNIST\nRoll No: 25I-2108",
          fontsize=14)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("MSE Loss", fontsize=12)
plt.xticks(range(1, epochs + 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()
print("Final training loss:", round(loss_history[-1], 6))

In [ ]:
_, _, _, _, _, A3_test = forward_pass(
    X_test, W1, b1, W2, b2, W3, b3)

predictions = np.argmax(A3_test, axis=1)
accuracy    = np.mean(predictions == Y_test) * 100

print("=" * 40)
print(f"  Test Accuracy : {accuracy:.2f}%")
print(f"  Correct       : {int(accuracy * 100)} / 10,000")
print("=" * 40)

print("\nPer-class accuracy:")
for digit in range(10):
    mask  = Y_test == digit
    acc_d = np.mean(predictions[mask] == digit) * 100
    print(f"  Digit {digit}: {acc_d:.1f}%")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
axes = axes.flatten()

for digit in range(10):
    # Find first test image whose true label is this digit
    idx        = np.where(Y_test == digit)[0][0]
    img        = X_test[idx].reshape(28, 28)
    pred_label = predictions[idx]
    true_label = Y_test[idx]
    confidence = A3_test[idx, pred_label] * 100

    axes[digit].imshow(img, cmap="gray")
    color = "green" if pred_label == true_label else "red"
    axes[digit].set_title(
        f"True: {true_label}  Pred: {pred_label}\n({confidence:.1f}%)",
        fontsize=10, color=color, fontweight="bold")
    axes[digit].axis("off")

plt.suptitle(
    "Sample Predictions — One Per Digit Class\nRoll No: 25I-2108",
    fontsize=13)
plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=150)
plt.show()
print("Green = correct   |   Red = wrong")

In [ ]:
print("="*50)
print("FINAL RESULTS SUMMARY")
print("="*50)
print(f"Roll Number: 251-2108")
print(f"Test Accuracy: {accuracy:.2f}%")
